**Статистическая языковая модель**

Евгений Борисов <esborisov@sevsu.ru>

подбираем наиболее вероятное продолжение цепочки слов (NLTK model)

In [1]:
import gzip
import requests
from bs4 import BeautifulSoup

In [2]:
# url='http://lib.ru/NEWPROZA/LOBAS/taxisty.txt'
# text = BeautifulSoup(requests.get(url).text).get_text()
# with gzip.open('taxisty.txt.gz','wt') as f: f.write(text)

# # with gzip.open('taxisty.txt.gz','rt') as f: text = f.read()

# text = text[1030:-7261].strip() # выкидываем заголовок и хвост страницы 
# print(f'символов:{len(text)}\n---------------\n'%())
# print(text[:343])

In [3]:
url='http://az.lib.ru/d/dostoewskij_f_m/text_0080.shtml'
text = BeautifulSoup(requests.get(url).text).get_text()
with gzip.open('dostoewskij.txt.gz','wt') as f: f.write(text)

# with gzip.open('dostoewskij.txt.gz','rt') as f: text = f.read()

text = text[2876:-664184].strip() # выкидываем заголовок и хвост страницы 
print(f'символов:{len(text)}\n---------------\n'%())
print(text[:355])

символов:1279540
---------------

Приступая к описанию недавних и столь странных событий, происшедших в нашем, доселе ничем не отличавшемся городе, я принужден, по неумению моему, начать несколько издалека, а именно некоторыми биографическими подробностями о талантливом и многочтимом Степане Трофимовиче Верховенском. Пусть эти подробности послужат лишь введением к предлагаемой хронике, 


In [4]:
from nltk import __version__ as nltk_version
print('nltk version:',nltk_version)

nltk version: 3.9.1


In [5]:
from nltk.tokenize import sent_tokenize as nltk_sentence_split
nltk_sentence_split?

Signature: nltk_sentence_split(text, language='english')
Docstring:
Return a sentence-tokenized copy of *text*,
using NLTK's recommended sentence tokenizer
(currently :class:`.PunktSentenceTokenizer`
for the specified language).

:param text: text to split into sentences
:param language: the model name in the Punkt corpus
File:      ~/.python_venv/default_cp313/lib/python3.13/site-packages/nltk/tokenize/__init__.py
Type:      function

In [6]:
from tqdm.auto import tqdm
from random import sample

from nltk.tokenize import sent_tokenize as nltk_sentence_split
from nltk.tokenize import word_tokenize as nltk_tokenize_word

sentences = [ 
    nltk_tokenize_word(s,language='russian') # разбиваем предложения на слова
    for s in tqdm(nltk_sentence_split(text,language='russian')) # режем текст на отдельные предложения
]

print('предложений: %i\n'%(len(sentences)))
display( sample(sentences,1) )

  0%|          | 0/14424 [00:00<?, ?it/s]

предложений: 14424



[['Я',
  'хочу',
  ',',
  'чтобы',
  'Дарья',
  'Павловна',
  'сама',
  'объявила',
  'мне',
  'из',
  'своих',
  'уст',
  'и',
  'пред',
  'лицом',
  'неба',
  ',',
  'или',
  'по',
  'крайней',
  'мере',
  'пред',
  'вами',
  '.']]

In [7]:
sentences = sentences[:1024*7] # ограничиваем датасет для ускорения процеса 

In [8]:
%%time

from nltk.lm.preprocessing import padded_everygram_pipeline 

ngram_len = 2

# генерируем учебный датасет
train, vocab = padded_everygram_pipeline(ngram_len, sentences)

CPU times: user 8 μs, sys: 0 ns, total: 8 μs
Wall time: 10.7 μs


In [9]:
# собираем модель

# from nltk.lm import MLE as LangModel 
from nltk.lm import Laplace as LangModel

model = LangModel(ngram_len) 
model.fit(train, vocab)

display(len(model.vocab))

20961

In [11]:
# генерируем продолжения
for sentence in sample(sentences,10): # выбираем рандомно 10 предложений
    if len(sentence)<10: continue
    # берём начало предложения
    sentence_ = sentence[:-(len(sentence)//4)]
    # генерируем возможные продолжения
    result = model.generate(3, text_seed=sentence_) 
    print(  ' '.join(sentence_)  + ' ... ' + str( result ) + '\n' )

Он медленно уселся на диван , на свое прежнее место в углу , и закрыл глаза , ... ['никак', 'нельзя-с.', 'Станешь']

- Только позвольте , Варвара Петровна , разве Степан Трофимович вам уже ... ['не', 'объявлял', ',']

- Господин Ставрогин , Николай Всеволодович ; мне вас на станции , едва лишь машина остановилась , ... ['чтобы', 'Nicolas', ',']

- начал он сам , осторожно смотря на Степана Трофимовича с ... ['одного', '!', '</s>']

Начну именно с восьмого дня после того воскресенья , то есть с понедельника вечером , потому что , в сущности , с этого вечера ... ['припас', 'себе', 'неоднократное']

* Мысль эта нравилась ; но большинство нашей светской молодежи выслушивало всё это с презрением и с видом самого ... ['начала', 'было', 'условлено']



In [12]:
# from nltk.util import bigrams
from nltk.util import ngrams

# оцениваем насколько хорошо модель предсказывает слова из датасета
text_ngrams = [ ng for s in sentences for ng in ngrams(s,ngram_len) ]

print( 'perplexity:', model.perplexity( text_ngrams ) )

perplexity: 4032.6262713096294


In [13]:
display( text_ngrams[:3] )

[('Приступая', 'к'), ('к', 'описанию'), ('описанию', 'недавних')]